# 07 — Khảo sát các cơ chế Readout Pooling cho MPNN (Quyết định D4)

## 1. Mục tiêu
Notebook này giải thích và trực quan hóa **Bước 7** trong quy trình XAI-Football:
- Khảo sát và đối sánh thực nghiệm 4 cơ chế gộp biểu diễn đồ thị (*Readout Pooling*) cho mô hình Message Passing Neural Network (MPNN):
  1. **Mean Pooling (`mean`):** Gộp trung bình toàn cục vector đặc trưng của mọi đỉnh trong đồ thị possession (mô hình gốc).
  2. **Global Max Pooling (`max`):** Gộp giá trị cực đại theo từng chiều đặc trưng trên toàn bộ đồ thị.
  3. **Attention Pooling (`attention`):** Gộp có trọng số học được thông qua mạng nơ-ron cơ chế chú ý (Gated Attention).
  4. **Mean kết hợp Last-edge Recipient (`mean_lastedge`):** Gộp trung bình kết hợp trực tiếp vector đặc trưng của cầu thủ nhận đường chuyền cuối cùng trước cú sút.
- Huấn luyện và đánh giá trên **5 seeds ngẫu nhiên cố định** (42, 123, 456, 789, 1024) theo đúng phân chia độc lập theo trận đấu (266 train / 57 validation / 57 test).
- Thực thi **Quyết định D4** theo nguyên tắc đăng ký trước: Chỉ thay thế MPNN Mean Pooling nguyên bản nếu biến thể mới cải thiện Validation AUC-PR tối thiểu $\ge 0.0100$.

## 2. Vị trí của bước trong pipeline
Đây là **Bước 7 — Khảo sát Readout Pooling MPNN (`scripts/evaluate_readouts.py`, `src/xai_football/modeling/gnn/readout.py`, `configs/model/mpnn_*.yaml`)**.

## 3. Input và nguồn dữ liệu
- Đồ thị PyG đã đóng gói: `data/processed/official/graphs.pt` (45.047 đồ thị possession).
- Phân chia trận đấu: `data/splits/official/train_matches.csv`, `val_matches.csv`, `test_matches.csv`.
- Cấu hình kiến trúc: `configs/model/mpnn.yaml`, `configs/model/mpnn_max.yaml`, `configs/model/mpnn_attn.yaml`, `configs/model/mpnn_lastedge.yaml`.
- Bảng kết quả chính thức: `results/tables/official/readout_comparison.csv` và manifest `readout_comparison_manifest.json`.

## 4. Config/script/module sử dụng
- Script thực thi: `scripts/evaluate_readouts.py`
- Module kiến trúc: `src/xai_football/modeling/gnn/readout.py`
- Lớp mô hình: `src/xai_football/modeling/gnn/mpnn.py`

## 5. Các bước xử lý & Đọc kết quả thực tế
Đoạn mã dưới đây nạp kết quả đối sánh 4 cơ chế Readout qua 5 seeds từ bảng kết quả chính thức và kiểm tra tính toàn vẹn.

In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display, Image

results_dir = Path("../results/tables/official")
readout_csv = results_dir / "readout_comparison.csv"
manifest_path = results_dir / "readout_comparison_manifest.json"

df_readout = pd.read_csv(readout_csv)
print(f"Nạp thành công {len(df_readout)} biến thể Readout Pooling từ: {readout_csv}")
display(df_readout)

if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)
    print("\n=== MANIFEST THÔNG TIN THỰC NGHIỆM ===")
    print(f"Baseline Backbone: {manifest.get('baseline_model')}")
    print(f"Best Validation Model: {manifest.get('best_val_model')}")
    print(f"Validation Gain: {manifest.get('val_auc_pr_gain'):+.4f}")
    print(f"Threshold Required: {manifest.get('threshold_required')}")
    print(f"Decision D4 Rule Applied: {manifest.get('decision_rule')}")

## 6. Kết quả và cách diễn giải

### 6.1. Trực quan hóa đối sánh Readout Pooling (Validation & Test AUC-PR)

In [ ]:
fig_path = Path("../results/figures/official/07_readout_comparison.png")
if fig_path.exists():
    display(Image(filename=str(fig_path), width=750))
else:
    print(f"Chưa tìm thấy biểu đồ tại: {fig_path}")

### 6.2. Diễn giải khoa học & Thực thi Quyết định D4
1. **Hiệu năng các cơ chế Readout:**
   - `mpnn` (Mean pooling): Validation AUC-PR = **0.5847 ± 0.0015**, Test AUC-PR = **0.5996 ± 0.0042**.
   - `mpnn_max` (Max pooling): Validation AUC-PR = 0.5624 ± 0.0031, Test AUC-PR = 0.5853 ± 0.0048.
   - `mpnn_attn` (Attention pooling): Validation AUC-PR = 0.5620 ± 0.0061, Test AUC-PR = 0.5820 ± 0.0074.
   - `mpnn_lastedge` (Mean kết hợp Last-edge): Validation AUC-PR = **0.5921 ± 0.0074**, Test AUC-PR = 0.5973 ± 0.0082.
2. **Quy tắc lựa chọn mô hình theo Quyết định D4:**
   - Biến thể `mpnn_lastedge` đạt Validation AUC-PR cao nhất, nhỉnh hơn baseline MPNN mean là $+0.0074$.
   - Tuy nhiên, quy tắc tiền nghiệm D4 yêu cầu mức cải thiện validation phải đạt tối thiểu **$\ge 0.0100$** mới tiến hành thay thế backbone chính thức.
   - Do $+0.0074 < +0.0100$, theo đúng đạo đức và nguyên tắc nghiên cứu khoa học, **mô hình MPNN Mean Pooling nguyên bản tiếp tục được duy trì làm backbone chính thức** cho toàn bộ pipeline XAI và Hybrid Centrality phía sau.

## 7. Kiểm tra chất lượng (Quality Checks)
Xác thực các tiêu chí chất lượng kỹ thuật của thực nghiệm Readout Pooling:

In [ ]:
checks = []
checks.append({
    "Tiêu chí": "Đủ 4 biến thể Readout Pooling",
    "Kết quả": "PASS" if len(df_readout) == 4 else "FAIL"
})
checks.append({
    "Tiêu chí": "Mỗi biến thể có đủ 5 seeds thực nghiệm",
    "Kết quả": "PASS" if (df_readout["n_seeds"] == 5).all() else "FAIL"
})
checks.append({
    "Tiêu chí": "Validation AUC-PR trong khoảng [0, 1]",
    "Kết quả": "PASS" if ((df_readout["val_auc_pr_mean"] >= 0) & (df_readout["val_auc_pr_mean"] <= 1)).all() else "FAIL"
})
checks.append({
    "Tiêu chí": "Quyết định D4: Backbone chính thức là mpnn (mean pooling)",
    "Kết quả": "PASS" if manifest.get("selected_final_model") == "mpnn" else "FAIL"
})

df_checks = pd.DataFrame(checks)
display(df_checks)
assert (df_checks["Kết quả"] == "PASS").all(), "Có kiểm tra chất lượng không đạt!"
print("Tất cả kiểm tra chất lượng đều ĐẠT (PASS)!")

## 8. Output được lưu ở đâu
- Bảng số liệu: `results/tables/official/readout_comparison.csv`
- Manifest thực nghiệm: `results/tables/official/readout_comparison_manifest.json`
- Biểu đồ: `results/figures/official/07_readout_comparison.png` và `.svg`
- Checkpoint các biến thể: `models/official/readout/`

## 9. Cách tái chạy
```bash
python scripts/evaluate_readouts.py --config configs/train.yaml
```

## 10. Giới hạn và giả định
- Thực nghiệm được tiến hành trên kiến trúc MPNN cố định 3 lớp message passing với 12 đặc trưng cạnh.
- Các mô hình GNN khác (GCN, GraphSAGE, GATv2) không được khảo sát cơ chế readout đa dạng do MPNN đã được chứng minh là kiến trúc vượt trội nhất ở Bước 6.
- Kết quả trung bình 5 seeds phản ánh tính ổn định vững chắc qua các lần khởi tạo trọng số.